# An acoustically shaped finite asphere

This notebook joins the apparatus, governing equations and **computed** optical evidence. The design is a 6 mm clear-aperture liquid optic with a 20 mm target focal distance. It is a numerical research result; the acoustic material data and several second-order mechanisms still need experimental calibration.

The full model contract is in `docs/model.md`. The time viewer uses physical states from the integrator, not an interpolation of the target.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from acoustic_freeform.single_interface.config import LensConfig
from acoustic_freeform.single_interface.surface import SurfaceSpace
from acoustic_freeform.single_interface.optics import best_fit_conic, spherical_optical_reference, trace_surface

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RESULT = ROOT / 'artifacts/studies/S01-single-interface/asphere'
report = json.loads((RESULT / 'report.json').read_text())
data = np.load(RESULT / 'trajectory.npz')
config = LensConfig(**report['configuration'])
space = SurfaceSpace(config)
final = data['coefficients'][-1]


## Apparatus and physical inputs

The 8 mm diameter chamber holds a finite resin volume above a flat optical window. Its sidewall carries 16 rows of 16 electrical sectors. Row sectors are tied for this axisymmetric experiment. The optical pupil is 6 mm in diameter, and the free surface is pinned at the 4 mm radius rim.

[Norland's NOA 61 data sheet](https://norlandproducts.com/wp-content/uploads/2025/02/Norland-Products-NOA-61-TDS.pdf) gives the liquid index 1.52, density 1231 kg/m³, viscosity 0.3 Pa·s and surface tension 0.04 N/m. Sound speed 1600 m/s and attenuation 5 Np/m are assumptions. The cured index 1.56 is not used.

In [ ]:
display(Image(filename=str(RESULT / 'figures/apparatus.png'), width=850))
print(f"Liquid volume: {report['liquid_volume_m3']*1e9:.3f} microlitres")
print(f"Bond number: {config.bond:.3f}")

## Why this particular surface is a lens

For parallel rays entering the liquid vertically, equal optical path to an axial focal point requires

$$n h(r)+\sqrt{[z_F-h(r)]^2+r^2}=\mathrm{constant}.$$

With vertex focal distance $f$, the refracting conic has $R=f(n-1)$ and $K=-n^2$:

$$z(r)=-\frac{r^2}{R+\sqrt{R^2+(n^2-1)r^2}}.$$

A height shift meets the rim. This defines the target and the fill volume, but does not constrain the evolving fluid to a conic. We fit a conic **after** the acoustic/fluid simulation and trace the actual surface slopes.

Fluid formation and optical performance must be evaluated together; see [Elgarisi et al. (2021)](https://doi.org/10.1364/OPTICA.438763) and [Na et al. (2024)](https://doi.org/10.1145/3680528.3687584).

In [ ]:
print('Fitted conic:', best_fit_conic(space, final))
print('Refocused spherical comparison:', spherical_optical_reference(space, final))
print('Computed best-focus ray RMS (um):', trace_surface(space, final)['rms_spot_at_best_focus_m']*1e6)

## What moves the liquid

The acoustic field is solved on the actual curved chamber:

$$\nabla^2P+(\omega/c+i\alpha)^2P=0,\qquad \mathbf V=\frac{\nabla P}{i\rho\omega}.$$

Peak phasors and a pressure-release surface give the radiation traction $\Pi=\rho|V_n|^2/4$. Coherent interference between array rows remains in this quadratic force. The pressure-release approximation replaces the much lower impedance air phase.

The liquid minimizes capillary/gravitational energy subject to conserved volume, while the array supplies work:

$$E=2\pi\sigma\int_0^a r\sqrt{1+h_r^2}\,dr+\pi\rho g\int_0^a r h^2\,dr,$$
$$V=\pi a^2d+2\pi\int_0^a r h\,dr.$$

The mobility is computed from a finite axisymmetric Stokes solve, including hoop strain, no-slip walls and a free surface. The surface evolves in a constant-volume modal subspace. Acoustic propagation is recalculated as that surface changes. [Chesneau et al. (2022)](https://doi.org/10.1103/PhysRevE.106.065104) provides the coupled radiation-stress/flow research basis; our finite-element apparatus and one-fluid approximation are different from their BEM model.

In [ ]:
times = data['times_s']
volume_error = [row['volume_error_m3'] for row in report['history']]
print('Maximum absolute volume error (m³):', max(map(abs, volume_error)))
print('Maximum convective Reynolds number:', max(row['flow_reynolds_number'] for row in report['history']))
print('Held-state source power (mW):', report['final']['source_power_w']*1000)

## Actual optical and time-domain result

The controller observes the full simulated surface and updates sixteen complex wall velocities every 20 ms. The fluid is integrated between these updates. The saved drive can also be replayed without the controller. At every time, optical metrics use the actual liquid shape and the same fixed detector plane.

In [ ]:
display(Image(filename=str(RESULT / 'figures/optical-validation.png'), width=1000))

## Independent numerical evidence

The acoustic boundary conditions and radiation force are checked against an independently derived Fourier–Bessel solution for a flat cylinder. Nonlinear capillarity is checked against an exact spherical cap in zero gravity. The final asphere is then checked by holding the physical array drive fixed while refining the acoustic mesh/order and surface basis.

In [ ]:
checks = json.loads((RESULT / 'verification/analytic-cavity.json').read_text())
print(checks)
spatial = json.loads((RESULT / 'spatial-convergence.json').read_text())
for row in spatial:
    print(f"P{row['order']}, {row['mesh']}, {row['modes']} modes: figure change {row['surface_change_rms_m']*1e9:.3f} nm RMS; ray spot {row['rms_spot_at_target_m']*1e6:.5f} um")
print(json.loads((RESULT / 'verification/time-convergence.json').read_text()))

## What this establishes, and what it leaves open

The numerical result is a reproducible asphere with a defined aperture and focus. It has substantially less geometric aberration than both the unforced finite liquid and a refocused spherical comparison. Its aspheric departure is resolved under spatial refinement, and a smaller-time-step replay preserves the held lens.

It does not yet predict a calibrated physical experiment or a cured optic. Acoustic speed/attenuation, transducer impedance, wetting, heating, streaming, cavitation and curing must be addressed. The Stokes approximation also limits quantitative fast-transient predictions. Full non-axisymmetric freeform surfaces require a 3D extension and azimuthally independent control.

Open `artifacts/studies/S01-single-interface/asphere/viewer/index.html` to inspect the complete apparatus and time evolution.